<a href="https://colab.research.google.com/github/Croop-weed/DeepLearning-parctice-/blob/main/data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install pypdf langchain-community langchain-text-splitters pdfplumber

In [26]:
import os
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument
import pdfplumber
from pypdf import PdfReader
import pandas as pd


In [27]:
path = "/content/SARA_BEN_2015_Report.pdf"

In [28]:
def extract_pdf_metadata(pdf_path):

    with open(pdf_path, 'rb') as f:
        reader = PdfReader(f)
        info = reader.metadata
        page_count = len(reader.pages)

    return {
        "title": info.title if info.title else os.path.basename(pdf_path),
        "author": info.author if info.author else "Unknown",
        "creator": info.creator if info.creator else "Unknown",
        "page_count": page_count
    }

In [29]:
print(extract_pdf_metadata(path))

{'title': 'SARA_BEN_2015_Report.pdf', 'author': 'Paulo', 'creator': 'Microsoft® Word\xa02013', 'page_count': 167}


In [30]:
def extract_content_and_tables(pdf_path):
    structured_pages = []
    full_text_list = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            text = page.extract_text(layout=False)

            page_text = text if text else ""
            full_text_list.append(page_text)

            structured_pages.append({
                "page_number": page_num,
                "text": page_text,
                "tables": tables if tables else []
            })

    entire_document_text = "\n--- PAGE BREAK ---\n".join(full_text_list)
    return entire_document_text, structured_pages

In [31]:
def decision_mind_pdf_loader(pdf_path, chunk_size=1000, chunk_overlap=150):

    metadata = extract_pdf_metadata(pdf_path)
    full_text, structured_pages = extract_content_and_tables(pdf_path)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    parent_doc = LCDocument(
        page_content=full_text,
        metadata={
            "source": os.path.basename(pdf_path),
            "title": metadata["title"]
        }
    )

    langchain_chunks = text_splitter.split_documents([parent_doc])

    return {
        "metadata": metadata,
        "full_text": full_text,
        "structured_pages": structured_pages,
        "vector_chunks": langchain_chunks
    }

In [34]:
pdf_file_path = path

# Create fake entity keys to mock the database setup
mock_decision_id = uuid4()
mock_user_id = uuid4()

# Run your new unified pipeline
pipeline_output = decision_mind_pdf_loader(pdf_file_path)

# Build payload for your SQLAlchemy 'Document' model instance
document_db_record = {
    "id": uuid4(),                                      # Document PK
    "decision_id": mock_decision_id,                    # FK to your Decision model
    "uploaded_by": mock_user_id,                        # FK to your User model
    "filename": os.path.basename(pdf_file_path),
    "stored_filename": f"{uuid4()}.pdf",
    "mime_type": "application/pdf",
    "file_path": pdf_file_path,
    "file_size": os.path.getsize(pdf_file_path) if os.path.exists(pdf_file_path) else 0,
    "document_type": "PDF",                             # Maps to your DocumentType Enum
    "extracted_text": pipeline_output["full_text"]       # Text stored securely in Postgres
}

print("✅ Text successfully loaded and split via LangChain!")
print(f"🔹 Total Characters Saved for DB: {len(pipeline_output['full_text'])}")
print(f"🔹 Number of LangChain Chunks prepared for Vector Store: {len(pipeline_output['vector_chunks'])}")
print("\n--- LangChain Chunk 1 Sample ---")
print(document_db_record)
print(pipeline_output['vector_chunks'][0].page_content[:300] + "...")

✅ Text successfully loaded and split via LangChain!
🔹 Total Characters Saved for DB: 284003
🔹 Number of LangChain Chunks prepared for Vector Store: 340

--- LangChain Chunk 1 Sample ---
{'id': UUID('b8fb03fe-51fe-4cdb-862c-c9db6bd90df4'), 'decision_id': UUID('cf0dfe91-10c4-4d8b-a598-d4dd69762809'), 'uploaded_by': UUID('2fb85bdf-450b-4a9d-ab51-878e33c97f2f'), 'filename': 'SARA_BEN_2015_Report.pdf', 'stored_filename': '0ed50b0d-8e3e-4aea-bc5d-ec434c9082ba.pdf', 'mime_type': 'application/pdf', 'file_path': '/content/SARA_BEN_2015_Report.pdf', 'file_size': 3594985, 'document_type': 'PDF', 'extracted_text': 'DIS PONIBILITE ET CA PACITE\nOPERATIONNELLE DES\nEnquête SARA\n2015\nSERVICES DE SANTE\n--- PAGE BREAK ---\nAvant-propos\nLa prise de décision sur la base des évidences est un élément essentiel dans la gestion efficace\ndu secteur en vue du renforcement de sa performance. Elle constitue une préoccupation\nconstante pour le Ministère de la Santé qui accorde une attention particulière au\